In [1]:
data_path = "../data/input.txt"
with open(data_path, 'r') as file:
    content = file.read()

content[:500]

"First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou are all resolved rather to die than to famish?\n\nAll:\nResolved. resolved.\n\nFirst Citizen:\nFirst, you know Caius Marcius is chief enemy to the people.\n\nAll:\nWe know't, we know't.\n\nFirst Citizen:\nLet us kill him, and we'll have corn at our own price.\nIs't a verdict?\n\nAll:\nNo more talking on't; let it be done: away, away!\n\nSecond Citizen:\nOne word, good citizens.\n\nFirst Citizen:\nWe are accounted poor"

In [ ]:
import re
import numpy as np

def preprocess_text(text):
    cleaned_text = re.sub(r"[^a-zA-Z\n\s]", "", text).lower()
    sentences = [sentence for sentence in cleaned_text.split("\n") if sentence.strip()]
    words = cleaned_text.replace("\n", " " ).split()

    stoi = {word: index for index, word in enumerate(sorted(set(words)), start=1)}
    itos = {index: word for word, index in stoi.items()}

    encoded_sentences = [[stoi[word] for word in sentence.split()] for sentence in sentences]
    input_sequence = [sentence[:i + 1] for sentence in encoded_sentences for i in range(1, len(sentence))]

    if not input_sequence:
        raise ValueError("Text must contain at least one sentence with at least two words.")

    max_len = max(len(sequence) for sequence in input_sequence)
    padded_encoded_sentences = np.zeros((len(input_sequence), max_len), dtype=int)

    for index, sequence in enumerate(input_sequence):
        padded_encoded_sentences[index, -len(sequence):] = sequence

    X = padded_encoded_sentences[:, :-1]
    y = padded_encoded_sentences[:, -1]
    num_classes = len(stoi) + 1

    return sentences, stoi, itos, encoded_sentences, input_sequence, padded_encoded_sentences, X, y, num_classes

In [ ]:
sentences, stoi, itos, encoded_sentences, input_sequence, padded_encoded_sentences, X, y, num_classes = preprocess_text(content)
sentences[:5]
X.shape

In [20]:
import torch
import torch.nn as nn

In [21]:
class LSTM(nn.Module):
    def __init__(self, vocab_size, hidden_layer, embedding_dim=100):
        super(LSTM, self).__init__()
        self.token_emb = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_layer,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_layer, vocab_size)

    def forward(self, X):
        X = X.long()
        x = self.token_emb(X)
        output, _ = self.lstm(x)
        logits = self.fc(output[:, -1, :])
        return logits

In [22]:
model = LSTM(vocab_size=num_classes, hidden_layer=128)
sample_batch = torch.tensor(X[:4], dtype=torch.long)
sample_logits = model(sample_batch)
sample_logits.shape

torch.Size([4, 12848])

In [28]:
from torch.utils.data import DataLoader, TensorDataset

dataset = TensorDataset(
    torch.tensor(X, dtype=torch.long),
    torch.tensor(y, dtype=torch.long),
)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 4
for epoch in range(epochs):
    model.train()
    total_loss = 0.0

    for xb, yb in loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch + 1}/{epochs} - loss: {total_loss / len(loader):.4f}")

Epoch 1/4 - loss: 6.0908
Epoch 2/4 - loss: 5.6426
Epoch 3/4 - loss: 5.2969
Epoch 4/4 - loss: 5.0005


In [34]:
def predict_next_words(text, model, stoi, itos, max_len, n_words=5):
    cleaned_text = re.sub(r"[^a-zA-Z\s]", "", text).lower()
    words = cleaned_text.split()
    encoded = [stoi[word] for word in words if word in stoi]

    if not encoded:
        raise ValueError("Input text must contain at least one word from the vocabulary.")

    generated_words = words.copy()
    model.eval()

    with torch.no_grad():
        for _ in range(n_words):
            current_encoded = encoded[-max_len:]
            padded = np.zeros((1, max_len), dtype=int)
            padded[0, -len(current_encoded):] = current_encoded

            logits = model(torch.tensor(padded, dtype=torch.long))
            next_word_id = torch.argmax(logits, dim=-1).item()
            next_word = itos.get(next_word_id, "<unk>")

            generated_words.append(next_word)
            encoded.append(next_word_id)

    return " ".join(generated_words)

test_input_text = 'romeo juliet'
generated_text = predict_next_words(test_input_text, model, stoi, itos, X.shape[1], n_words=10)
print(f"Input: {test_input_text}")
print(f"Generated text: {generated_text}")

Input: romeo juliet
Generated text: romeo juliet i have been drinking and so i am a king
